# Evaluating LLM Applications Using LangChain

Complete updated companion, based on the author-supplied LangChain book notebook. Run top to bottom with a locally configured OpenAI key. Calls incur API charges. The career Markdown is an original teaching fixture, not the draft ebook. Original archive code and transcripts remain in the nonexecuting appendix.

## Notebook setup

In [ ]:
import os,sys,subprocess,importlib.metadata,hashlib,urllib.request
from pathlib import Path
from tempfile import TemporaryDirectory
dependencies = {'pandas': '2.3.3', 'numpy': '2.3.3', 'openai': '2.43.0', 'langchain': '1.2.12', 'langchain-openai': '1.1.11', 'langchain-core': '1.4.8', 'langgraph': '1.1.2', 'langchain-community': '0.4.1', 'langchain-text-splitters': '1.1.1', 'ddgs': '9.14.4', 'python-dotenv': '1.2.2', 'pydantic': '2.13.4', 'httpx': '0.28.1', 'certifi': '2025.11.12'}
missing = []
for name,version in dependencies.items():
    try:
        if importlib.metadata.version(name) != version: missing.append(f'{name}=={version}')
    except importlib.metadata.PackageNotFoundError:
        missing.append(f'{name}=={version}')
if missing:
    subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY locally before running this notebook. Never paste it into a cell.')
os.environ.setdefault('MODEL','gpt-5.4-mini')
os.environ.setdefault('EMBEDDING_MODEL','text-embedding-3-small')
workspace = TemporaryDirectory(prefix='tdb-langchain-12-')
work = Path(workspace.name)
source_revision = '5984620e5691a3991d5e9353bc2cf2533e576234'
files = {'product-reviews.csv': 'ce8895ea5d502554d8da2c7fc6b20e6e9591816ee96e66a8ad329bec7c009522', 'career-in-ai.md': 'bbe0f834b947e46a5e4d152119ee76bb7ed784820d7e057c06c8f1e3b0a3971c'}
for name,digest in files.items():
    url = f'https://raw.githubusercontent.com/youssefHosni/hands-on-langchain/{source_revision}/data/{name}'
    with urllib.request.urlopen(url,timeout=60) as response: payload = response.read()
    assert hashlib.sha256(payload).hexdigest() == digest, name
    (work/name).write_bytes(payload)
print('Python:',sys.version.split()[0])
print('Configured chat model:',os.environ['MODEL'])
print('Pinned example CSV and Markdown inputs verified; no draft ebook is used.')

## Build a QA system to evaluate

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(find_dotenv(usecwd=True))
llm_model = os.getenv("MODEL", "gpt-5.4-mini")
embedding_model = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

In [ ]:
# Load the document, split it into chunks, embed them into an in-memory vector store
text = (work / "career-in-ai.md").read_text()
docs = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80).create_documents([text])

embeddings = OpenAIEmbeddings(model=embedding_model, max_retries=0)
vectorstore = InMemoryVectorStore.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"{len(docs)} chunks embedded")

Now a retrieval-QA chain in plain LCEL: fetch relevant chunks, stuff them into the prompt, answer.

In [ ]:
llm = ChatOpenAI(temperature=0.0, model=llm_model, max_tokens=2048, max_retries=0, reasoning_effort="none")

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

qa_prompt = ChatPromptTemplate.from_template(
    "Use the following context to answer the question. "
    "If you don't know, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | qa_prompt | llm | StrOutputParser()
)
qa_chain.invoke("Who founded DeepLearning.AI?")

## Manual evaluation

The simplest approach: hand-write a few questions with known answers (ground truth) and eye-ball the
chain's responses. It is honest and quick, but it does not scale.

In [ ]:
examples = [
    {"query": "Is machine learning foundations the most important skill?", "answer": "Yes"},
    {"query": "What are the Python frameworks you need to learn?", "answer": "TensorFlow and PyTorch"},
]

print(qa_chain.invoke(examples[0]["query"]))

## LLM-assisted evaluation

Writing test cases by hand is slow. We can have an LLM **generate** question/answer pairs from the
documents, and then have an LLM **grade** the system's answers. Legacy `QAGenerateChain` and `QAEvalChain` live in `langchain-classic`. Here we build both workflows with `with_structured_output`.

### Generate question/answer pairs from the documents

In [ ]:
from pydantic import BaseModel, Field

class QAPair(BaseModel):
    """A question and its correct answer, grounded in a document."""
    query: str = Field(description="A question answerable from the document.")
    answer: str = Field(description="The correct answer, grounded in the document.")

gen_prompt = ChatPromptTemplate.from_template(
    "You are a teacher writing a quiz. From the document below, write one question and its "
    "correct answer.\n\nDocument:\n{doc}"
)
qa_generator = gen_prompt | llm.with_structured_output(QAPair)

# generate a QA pair from each of the first few chunks
generated = qa_generator.batch([{"doc": d.page_content} for d in docs[:4]])
for g in generated:
    print("Q:", g.query)
    print("A:", g.answer, "\n")

Add the generated pairs to our hand-written ones to form the full evaluation set.

In [ ]:
examples += [{"query": g.query, "answer": g.answer} for g in generated]
print(f"{len(examples)} examples total")

### Run the QA chain on every example

In [ ]:
predictions = qa_chain.batch([e["query"] for e in examples])
for e, p in zip(examples, predictions):
    print("Q:", e["query"])
    print("Predicted:", p[:120], "...\n")

### Grade the predictions with an LLM judge

We give the judge the question, the ground-truth answer, and the predicted answer, and ask for a
`CORRECT` / `INCORRECT` grade plus a one-line reason.

In [ ]:
from typing import Literal

class Grade(BaseModel):
    """Grade of a predicted answer against the ground truth."""
    grade: Literal["CORRECT", "INCORRECT"] = Field(description="Is the prediction correct?")
    reasoning: str = Field(description="One sentence explaining the grade.")

judge_prompt = ChatPromptTemplate.from_template(
    "You are grading a quiz. Mark the student's answer CORRECT if it is factually consistent with "
    "the true answer, even if worded differently; otherwise INCORRECT.\n\n"
    "Question: {query}\nTrue answer: {answer}\nStudent answer: {prediction}"
)
judge = judge_prompt | llm.with_structured_output(Grade)

graded = judge.batch([
    {"query": e["query"], "answer": e["answer"], "prediction": p}
    for e, p in zip(examples, predictions)
])

In [ ]:
for i, (e, p, g) in enumerate(zip(examples, predictions, graded)):
    print(f"Example {i} — {g.grade}")
    print("  Question :", e["query"])
    print("  Real     :", e["answer"])
    print("  Predicted:", p[:100], "...")
    print("  Why      :", g.reasoning, "\n")

## Observing behind the scenes

A final answer alone rarely tells you *why* something went wrong — especially in retrieval, where the
fault is often the fetched context, not the model. `set_debug(True)` makes LangChain print every step:
the inputs, the retrieved context, the exact prompt sent to the model, and the response.

In [ ]:
from langchain_core.globals import set_debug

set_debug(True)
_ = qa_chain.invoke("What does Andrew Ng compare AI to?")
set_debug(False)   # turn it back off

Each model response also carries metadata, including token usage — useful for tracking cost over
time. Calling the model directly (rather than through `StrOutputParser`) exposes it:

In [ ]:
resp = llm.invoke("What does Andrew Ng compare AI to? Answer in one sentence.")
print(resp.content)
print("token usage:", resp.usage_metadata)

> For team-scale evaluation — datasets, automatic scoring, and tracking experiments over time —
> LangChain's hosted companion **LangSmith** provides tracing and evaluators. The manual, generated,
> and LLM-judge techniques in this chapter are the same ideas it automates.

## Regression checks

These checks verify data flow and contracts. Explicit scripted probes test control flow only; they are separate from the live model calls above.

In [ ]:
import json
assert len(docs)==7 and len(examples)==len(predictions)==len(graded)==6
assert len(generated)==4 and all(g.query and g.answer for g in generated)
assert all(isinstance(p,str) and p.strip() for p in predictions)
assert all(g.grade in {'CORRECT','INCORRECT'} and g.reasoning for g in graded)
for label in ['CORRECT','INCORRECT']: assert Grade(grade=label,reasoning='test').grade==label
try: Grade(grade='MAYBE',reasoning='test')
except ValueError: pass
else: raise AssertionError('Invalid grade accepted')
query_vector=embeddings.embed_query('Who founded DeepLearning.AI?')
import numpy as np
q=np.asarray(query_vector); q=q/np.linalg.norm(q)
stored=list(vectorstore.store.values())
scores=[float(np.dot(q,np.asarray(d['vector'])/np.linalg.norm(d['vector'])))for d in stored]
expected=[stored[i]['text']for i in np.argsort(scores)[::-1][:3]]
actual=[d.page_content for d in retriever.invoke('Who founded DeepLearning.AI?')]
assert actual==expected
print(json.dumps({'chunks':len(docs),'examples':len(examples),'model_grades':[g.grade for g in graded],'cosine_ranking':'independently verified','checks':'passed'}))

## Original archive appendix

The following is the complete original article text and all its original code/transcripts, preserved for reference. Fenced examples here are not executed. The maintained tutorial above follows the newer author-supplied notebook.

Hands-On LangChain for LLM Application Development

When constructing a sophisticated application employing an LLM, a crucial yet challenging aspect revolves around evaluating its performance. How can you ascertain if it meets accuracy standards?

Moreover, if you opt to alter your implementation — perhaps by substituting a different LLM or adjusting the strategy for utilizing a vector database or other retrieval mechanisms — how can you gauge whether these changes enhance or detract from the application?

This article discusses the challenges of evaluating the performance of applications built with large language models (LLMs) and explores strategies for effectively assessing their accuracy and effectiveness.

It emphasizes the importance of understanding the inputs and outputs of each step in the application’s workflow and introduces frameworks and tools designed to aid in evaluation.

Additionally, it explores the concept of using language models and chains themselves to evaluate other models and applications. With the rise of prompt-based development and the growing reliance on LLMs, the process of evaluating application workflows is undergoing reevaluation.

## Setting Up Working Environment

As usual, we will start by setting the working environment and importing the important packages and libraries we will use throughout this article. First, we will define the environment variables that will be used later to load the OpenAI API key to use the LLM.

````python
import os


from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

llm_model = "gpt-3.5-turbo-0301"
````

We need to have a chain to evaluate so we will use the document question-answering chain. To do this we will have to import the packages that we are going to use and then load the data using LangChain PyDFloader We are going to import everything we need.

````python
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import CSVLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch
from langchain.document_loaders import PyPDFLoader

file = 'eBook-How-to-Build-a-Career-in-AI.pdf'
loader = PyPDFLoader(file_path=file)
data = loader.load()
````

We’re going to create that index with one line, and then we’re going to create the retrieval QA chain by specifying the language model, the chain type, the retriever, and then the verbosity that we’re going to print out.

````python
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch
).from_loaders([loader])

llm = ChatOpenAI(temperature = 0.0, model=llm_model)
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=index.vectorstore.as_retriever(), 
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
````

Now that we have our application set, we need to figure out what are some data points that we want to evaluate it on.

## Manual Evaluation & Debugging

We will start with the simplest method of debugging, in which we will come up with data points that we think are good examples and test them. We can just look at some of the data and come up with example questions and then example ground truth answers that we can later use to evaluate.

So if we look at a few of the documents here, We can get a sense of what’s going on inside them. We are using the How to Build a Career in AI document by Andrew NG.

````python
data[10]
````

````text
Document(page_content=’PAGE 11\nThe Best Way to Build \na New Habit\nOne of my favorite books is BJ Fogg’s, Tiny Habits: The Small Changes That Change \nEverything. Fogg explains that the best way to build a new habit is to start small \nand succeed, rather than start too big and fail. For example, rather than trying to \nexercise for 30 minutes a day, he recommends aspiring to do just one push-up, and \ndoing it consistently.\nThis approach may be helpful to those of you who want to spend more time studying. \nIf you start by holding yourself accountable for watching, say, 10 seconds of an \neducational video every day — and you do so consistently — the habit of studying daily \nwill grow naturally. Even if you learn nothing in that 10 seconds, you’re establishing the \nhabit of studying a little every day. On some days, maybe you’ll end up studying for an \nhour or longer.’, metadata={‘source’: ‘eBook-How-to-Build-a-Career-in-AI.pdf’, ‘page’: 10}
````

This part of the document discusses th importance of continuous learning and developing the habit of learning.

````python
data[8]
````

````text
Document(page_content=’PAGE 9In the previous chapter, I introduced three key steps for building a career in AI: learning \nfoundational technical skills, working on projects, and finding a job, all of which is supported \nby being part of a community. In this chapter, I’d like to dive more deeply into the first step: \nlearning foundational skills.\nMore research papers have been published on AI than anyone can read in a lifetime. So, when \nlearning, it’s critical to prioritize topic selection. I believe the most important topics for a technical \ncareer in machine learning are:\nFoundational machine learning skills: For example, it’s important to understand models such \nas linear regression, logistic regression, neural networks, decision trees, clustering, and anomaly \ndetection. Beyond specific models, it’s even more important to understand the core concepts \nbehind how and why machine learning works, such as bias/variance, cost functions, regularization, \noptimization algorithms, and error analysis.\nDeep learning: This has become such a large fraction of machine learning that it’s hard to excel \nin the field without some understanding of it! It’s valuable to know the basics of neural networks, \npractical skills for making them work (such as hyperparameter tuning), convolutional networks, \nsequence models, and transformers.\nSoftware development: While you can get a job and make huge contributions with only machine \nlearning modeling skills, your job opportunities will increase if you can also write good software \nto implement complex AI systems. These skills include programming fundamentals, data \nstructures (especially those that relate to machine learning, such as data frames), algorithms \n(including those related to databases and data manipulation), software design, familiarity with \nPython, and familiarity with key libraries such as TensorFlow or PyTorch, and scikit-learn.Math relevant to machine learning: Key areas include linear algebra (vectors, matrices, and various \nmanipulations of them) as well as probability and statistics (including discrete and continuous \nprobability, standard probability distributions, basic rules such as independence and Bayes’ rule, \nand hypothesis testing). In addition, exploratory data analysis (EDA) — using visualizations and other \nmethods to systematically explore a dataset — is an underrated skill. I’ve found EDA particularly \nuseful in data-centric AI development, where analyzing errors and gaining insights can really help \ndrive progress! Finally, a basic intuitive understanding of calculus will also help. The math needed \nto do machine learning well has been changing. For instance, although some tasks require calculus, \nimproved automatic differentiation software makes it possible to invent and implement new neural \nnetwork architectures without doing any calculus. This was almost impossible a decade ago.Learning Technical Skills For a Promising AI Career CHAPTER 2', metadata={‘source’: ‘eBook-How-to-Build-a-Career-in-AI.pdf’, ‘page’: 8})
````

The second page explores the different skills you need to start a career in AI and data science. So the first one we can ask is a simple question Is machine learning foundations the most important skill?

for the second one, we can ask what are the Python frameworks you need to learn. The answer to this question is in the second document.

````python
examples = [
    {
        "query": "Is machine learning foundations the most important skill?",
        "answer": "Yes"
    },
    {
        "query": "What are Python frameworks you need to learn ?",
        "answer": "Tensorflow and PyTorch"
    }
]
````

But this doesn’t scale that. It takes a bit of time to look through each example and figure out what’s going on. So a better way to do that is to automate it.

## LLM-Assisted Evaluation

One of the methods that we can use to automate the evaluation process is with LLM themselves. We have a chain in Langchain that can do exactly that. So we can import the QA generation chain, and this will take in documents and will create a question-answer pair from each document. It’ll do this using a language model itself. So we need to create this chain by passing in the chat open AI language model.

````python
from langchain.evaluation.qa import QAGenerateChain
````

Then from there, we can create a list of examples.

````python
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(model=llm_model))
````

We are going to use the apply and parse method because this is applying an output parser to the result because we want to get back a dictionary that has the query and answer pair, not just a single string.

````python
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)
new_examples[0]
````

````text
new_examples[0]
new_examples[0]
{‘query’: ‘Who is the founder of DeepLearning.AI and what insights did he provide in the document?’,
 ‘answer’: ‘The founder of DeepLearning.AI is Andrew Ng and he provided insights on how to build a career in AI in the document “How to Build Your Career in AI: A Simple Guide.”’}
````

And so now if we look at what exactly is returned here, we can see a query and we can see an answer. Let’s check the document that this is a question and answer for.

````python
data[0]
````

````text
Document(page_content=’PAGE 1Founder, DeepLearning.AICollected Insights\nfrom Andrew Ng\nHow to \nBuild\nYour\nCareer\nin AIA Simple Guide\n’, metadata={‘source’: ‘eBook-How-to-Build-a-Career-in-AI.pdf’, ‘page’: 0})
````

We just generated a bunch of question-answer pairs. We won’t have to write it all ourselves which will save us a lot of time and we can do more testing cases. Now let’s go ahead and add these examples into the examples that we already created.

````python
examples += new_examples
````

So we got these examples now, but how exactly do we evaluate what’s going on? The first thing we want to do is just run an example through the chain and take a look at the output it produces.

````python
qa.run(examples[0]["query"])
````

````text
> Entering new RetrievalQA chain…
````

````text
> Finished chain.
‘According to the context provided, learning foundational technical skills, including machine learning foundations, is one of the three key steps for building a career in AI. However, it is not explicitly stated that it is the most important skill. It is important to prioritize topic selection and have a good understanding of core concepts behind how and why machine learning works. Additionally, skills in deep learning, software development, and math relevant to machine learning are also important.’
````

When we input a query, we receive an answer. However, this approach limits our visibility into the chain’s inner workings. What exact prompt is fed into the language model? Which documents does it fetch?

In more complex chains with multiple steps, what intermediate results are generated? Simply observing the final answer often isn’t sufficient for understanding potential issues within the chain. To address this, we have a helpful utility in Lane Chain called Lane Chain Debug.

## Observing Behind the Scenes

If we want to observe what is happening behind the scenes we can set the LangChain debug equals to true, and we now rerun the same example as above, we can see that it starts printing out a lot more information.

````python
import langchain
langchain.debug = True

qa.run(examples[0]["query"])
````

````text
[chain/start] [1:chain:RetrievalQA] Entering Chain run with input:
{
 “query”: “Is machine learning foundations the most important skill?”
}
[chain/start] [1:chain:RetrievalQA > 2:chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [1:chain:RetrievalQA > 2:chain:StuffDocumentsChain > 3:chain:LLMChain] Entering Chain run with input:
{
 “question”: “Is machine learning foundations the most important skill?”,
 “context”: “PAGE 9In the previous chapter, I introduced three key steps for building a career in AI: learning \nfoundational technical skills, working on projects, and finding a job, all of which is supported \nby being part of a community. In this chapter, I\u2019d like to dive more deeply into the first step: \nlearning foundational skills.\nMore research papers have been published on AI than anyone can read in a lifetime. So, when \nlearning, it\u2019s critical to prioritize topic selection. I believe the most important topics for a technical \ncareer in machine learning are:\nFoundational machine learning skills: For example, it\u2019s important to understand models such \nas linear regression, logistic regression, neural networks, decision trees, clustering, and anomaly \ndetection. Beyond specific models, it\u2019s even more important to understand the core concepts \nbehind how and why machine learning works, such as bias/variance, cost functions, regularization, \noptimization algorithms, and error analysis.<<<<>>>>>Deep learning: This has become such a large fraction of machine learning that it\u2019s hard to excel \nin the field without some understanding of it! It\u2019s valuable to know the basics of neural networks, \npractical skills for making them work (such as hyperparameter tuning), convolutional networks, \nsequence models, and transformers.\nSoftware development: While you can get a job and make huge contributions with only machine \nlearning modeling skills, your job opportunities will increase if you can also write good software \nto implement complex AI systems. These skills include programming fundamentals, data \nstructures (especially those that relate to machine learning, such as data frames), algorithms \n(including those related to databases and data manipulation), software design, familiarity with \nPython, and familiarity with key libraries such as TensorFlow or PyTorch, and scikit-learn.Math relevant to machine learning: Key areas include linear algebra (vectors, matrices, and various<<<<>>>>>PAGE 13Should you Learn Math to Get a Job in AI? CHAPTER 3\nIs math a foundational skill for AI? It\u2019s always nice to know more math! But there\u2019s so much to \nlearn that, realistically, it\u2019s necessary to prioritize. Here\u2019s how you might go about strengthening \nyour math background.\nTo figure out what\u2019s important to know, I find it useful to ask what you need to know to make \nthe decisions required for the work you want to do. At DeepLearning.AI, we frequently ask, \n\u201cWhat does someone need to know to accomplish their goals?\u201d The goal might be building a \nmachine learning model, architecting a system, or passing a job interview.\nUnderstanding the math behind algorithms you use is often helpful, since it enables you to \ndebug them. But the depth of knowledge that\u2019s useful changes over time. As machine learning \ntechniques mature and become more reliable and turnkey, they require less debugging, and a \nshallower understanding of the math involved may be sufficient to make them work.<<<<>>>>>PAGE 10This is a lot to learn!\nEven after you master everything on this list, I hope you\u2019ll keep learning and continue to deepen \nyour technical knowledge. I\u2019ve known many machine learning engineers who benefitted from \ndeeper skills in an application area such as natural language processing or computer vision, or in \na technology area such as probabilistic graphical models or building scalable software systems.\nHow do you gain these skills? There\u2019s a lot of good content on the internet, and in theory, \nreading dozens of web pages could work. But when the goal is deep understanding, reading \ndisjointed web pages is inefficient because they tend to repeat each other, use inconsistent \nterminology (which slows you down), vary in quality, and leave gaps. That\u2019s why a good course \n\u2014 in which a body of material has been organized into a coherent and logical form \u2014 is often the \nmost time-efficient way to master a meaningful body of knowledge. When you\u2019ve absorbed the”
}
[llm/start] [1:chain:RetrievalQA > 2:chain:StuffDocumentsChain > 3:chain:LLMChain > 4:llm:ChatOpenAI] Entering LLM run with input:
{
 “prompts”: [
 “System: Use the following pieces of context to answer the users question. \nIf you don’t know the answer, just say that you don’t know, don’t try to make up an answer.\n — — — — — — — — \nPAGE 9In the previous chapter, I introduced three key steps for building a career in AI: learning \nfoundational technical skills, working on projects, and finding a job, all of which is supported \nby being part of a community. In this chapter, I\u2019d like to dive more deeply into the first step: \nlearning foundational skills.\nMore research papers have been published on AI than anyone can read in a lifetime. So, when \nlearning, it\u2019s critical to prioritize topic selection. I believe the most important topics for a technical \ncareer in machine learning are:\nFoundational machine learning skills: For example, it\u2019s important to understand models such \nas linear regression, logistic regression, neural networks, decision trees, clustering, and anomaly \ndetection. Beyond specific models, it\u2019s even more important to understand the core concepts \nbehind how and why machine learning works, such as bias/variance, cost functions, regularization, \noptimization algorithms, and error analysis.<<<<>>>>>Deep learning: This has become such a large fraction of machine learning that it\u2019s hard to excel \nin the field without some understanding of it! It\u2019s valuable to know the basics of neural networks, \npractical skills for making them work (such as hyperparameter tuning), convolutional networks, \nsequence models, and transformers.\nSoftware development: While you can get a job and make huge contributions with only machine \nlearning modeling skills, your job opportunities will increase if you can also write good software \nto implement complex AI systems. These skills include programming fundamentals, data \nstructures (especially those that relate to machine learning, such as data frames), algorithms \n(including those related to databases and data manipulation), software design, familiarity with \nPython, and familiarity with key libraries such as TensorFlow or PyTorch, and scikit-learn.Math relevant to machine learning: Key areas include linear algebra (vectors, matrices, and various<<<<>>>>>PAGE 13Should you Learn Math to Get a Job in AI? CHAPTER 3\nIs math a foundational skill for AI? It\u2019s always nice to know more math! But there\u2019s so much to \nlearn that, realistically, it\u2019s necessary to prioritize. Here\u2019s how you might go about strengthening \nyour math background.\nTo figure out what\u2019s important to know, I find it useful to ask what you need to know to make \nthe decisions required for the work you want to do. At DeepLearning.AI, we frequently ask, \n\u201cWhat does someone need to know to accomplish their goals?\u201d The goal might be building a \nmachine learning model, architecting a system, or passing a job interview.\nUnderstanding the math behind algorithms you use is often helpful, since it enables you to \ndebug them. But the depth of knowledge that\u2019s useful changes over time. As machine learning \ntechniques mature and become more reliable and turnkey, they require less debugging, and a \nshallower understanding of the math involved may be sufficient to make them work.<<<<>>>>>PAGE 10This is a lot to learn!\nEven after you master everything on this list, I hope you\u2019ll keep learning and continue to deepen \nyour technical knowledge. I\u2019ve known many machine learning engineers who benefitted from \ndeeper skills in an application area such as natural language processing or computer vision, or in \na technology area such as probabilistic graphical models or building scalable software systems.\nHow do you gain these skills? There\u2019s a lot of good content on the internet, and in theory, \nreading dozens of web pages could work. But when the goal is deep understanding, reading \ndisjointed web pages is inefficient because they tend to repeat each other, use inconsistent \nterminology (which slows you down), vary in quality, and leave gaps. That\u2019s why a good course \n\u2014 in which a body of material has been organized into a coherent and logical form \u2014 is often the \nmost time-efficient way to master a meaningful body of knowledge. When you\u2019ve absorbed the\nHuman: Is machine learning foundations the most important skill?”
 ]
}
[llm/end] [1:chain:RetrievalQA > 2:chain:StuffDocumentsChain > 3:chain:LLMChain > 4:llm:ChatOpenAI] [964.117ms] Exiting LLM run with output:
{
 “generations”: [
 [
 {
 “text”: “According to the context provided, learning foundational technical skills, including machine learning foundations, is one of the three key steps for building a career in AI. However, it is not explicitly stated that it is the most important skill. It is important to prioritize topic selection and have a good understanding of core concepts behind how and why machine learning works. Additionally, skills in deep learning, software development, and math relevant to machine learning are also important.”,
 “generation_info”: null,
 “message”: {
 “content”: “According to the context provided, learning foundational technical skills, including machine learning foundations, is one of the three key steps for building a career in AI. However, it is not explicitly stated that it is the most important skill. It is important to prioritize topic selection and have a good understanding of core concepts behind how and why machine learning works. Additionally, skills in deep learning, software development, and math relevant to machine learning are also important.”,
 “additional_kwargs”: {},
 “example”: false
 }
 }
 ]
 ],
 “llm_output”: {
 “token_usage”: {
 “prompt_tokens”: 869,
 “completion_tokens”: 88,
 “total_tokens”: 957
 },
 “model_name”: “gpt-3.5-turbo-0301”
 }
}
[chain/end] [1:chain:RetrievalQA > 2:chain:StuffDocumentsChain > 3:chain:LLMChain] [964.73ms] Exiting Chain run with output:
{
 “text”: “According to the context provided, learning foundational technical skills, including machine learning foundations, is one of the three key steps for building a career in AI. However, it is not explicitly stated that it is the most important skill. It is important to prioritize topic selection and have a good understanding of core concepts behind how and why machine learning works. Additionally, skills in deep learning, software development, and math relevant to machine learning are also important.”
}
[chain/end] [1:chain:RetrievalQA > 2:chain:StuffDocumentsChain] [965.186ms] Exiting Chain run with output:
{
 “output_text”: “According to the context provided, learning foundational technical skills, including machine learning foundations, is one of the three key steps for building a career in AI. However, it is not explicitly stated that it is the most important skill. It is important to prioritize topic selection and have a good understanding of core concepts behind how and why machine learning works. Additionally, skills in deep learning, software development, and math relevant to machine learning are also important.”
}
[chain/end] [1:chain:RetrievalQA] [1.16s] Exiting Chain run with output:
{
 “result”: “According to the context provided, learning foundational technical skills, including machine learning foundations, is one of the three key steps for building a career in AI. However, it is not explicitly stated that it is the most important skill. It is important to prioritize topic selection and have a good understanding of core concepts behind how and why machine learning works. Additionally, skills in deep learning, software development, and math relevant to machine learning are also important.”
}
‘According to the context provided, learning foundational technical skills, including machine learning foundations, is one of the three key steps for building a career in AI. However, it is not explicitly stated that it is the most important skill. It is important to prioritize topic selection and have a good understanding of core concepts behind how and why machine learning works. Additionally, skills in deep learning, software development, and math relevant to machine learning are also important.’
````

When examining the output closely, we notice it first delves into the retrieval QA chain, followed by the documents chain. As mentioned, the method is utilized here. Subsequently, it enters the LLM chain, where various inputs are involved. The original question is evident, alongside the provided context, synthesized from multiple retrieved documents.

In question-answering scenarios, errors often arise not from the language model itself, but from flaws in the retrieval process. Thus, scrutinizing the question and context aids in debugging. Further inspection reveals the inputs into the language model, Chat OpenAI itself.

Here, we have the complete prompt passed in, comprising a system message and the prompt description utilized by the question-answering chain under the hood, which we haven’t explored until now.

The prompt instructs to utilize specific context pieces for answering the user’s question and emphasizes not to fabricate answers if uncertain. Following this, we encounter the inserted context and the human question posed.

Moreover, detailed information about the return type is provided, including token usage metrics such as prompt tokens, completion tokens, total tokens, and the model name. This data proves valuable for monitoring token usage in chains or language model calls over time, correlating closely with the total cost.

Let's ask a language model to do it. First, we need to create predictions for all the examples. Then I’m going to create predictions for all the different examples. We are going to loop through these chains, getting a prediction for each one.

````python
predictions = qa.apply(examples)
````

````text
> Entering new RetrievalQA chain…
````

````text
> Finished chain.
````

````text
> Entering new RetrievalQA chain…
````

````text
> Finished chain.
````

````text
> Entering new RetrievalQA chain…
````

````text
> Finished chain.
````

````text
> Entering new RetrievalQA chain…
````

````text
> Finished chain.
````

````text
> Entering new RetrievalQA chain…
````

````text
> Finished chain.
````

````text
> Entering new RetrievalQA chain…
````

````text
> Finished chain.
````

````text
> Entering new RetrievalQA chain…
````

````text
> Finished chain
````

With these examples in hand, let’s consider evaluating them. First, we’ll import the QA question answering eval chain. Then, we’ll instantiate this chain with a language model, as we’ll utilize it for evaluation purposes. Subsequently, we’ll invoke the ‘evaluate’ function on this chain, passing in examples and predictions, and receive graded outputs in return.

````python
llm = ChatOpenAI(temperature=0, model=llm_model)
eval_chain = QAEvalChain.from_llm(llm)

graded_outputs = eval_chain.evaluate(examples, predictions)
````

To observe the process for each example, we’ll iterate through them. We’ll print the question, which was generated by a language model, followed by the real answer, also generated by a language model with access to the complete document.

Next, we’ll display the predicted answer, generated by a language model within the QA chain, utilizing embeddings and vector databases for retrieval before passing through the language model for prediction.

Additionally, we’ll print the grade, determined by a language model when prompted to evaluate and assess correctness. By systematically looping through these examples and printing their details, we gain insight into each example’s evaluation.

````python
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i]['text'])
    print()
````

````text
Example 0:
Question: Is machine learning foundations the most important skill?
Real Answer: Yes
Predicted Answer: According to the context provided, learning foundational technical skills, including machine learning foundations, is one of the three key steps for building a career in AI. However, it is not explicitly stated that it is the most important skill. It is important to prioritize topic selection and have a good understanding of core concepts behind how and why machine learning works. Additionally, skills in deep learning, software development, and math relevant to machine learning are also important.
Predicted Grade: INCORRECT
````

````text
Example 1:
Question: What are Python frameworks you need to learn ?
Real Answer: Tensorflow and PyTorch
Predicted Answer: The given context does not provide a comprehensive list of Python frameworks that one needs to learn. However, it does mention some key libraries such as TensorFlow, PyTorch, and scikit-learn that are commonly used in machine learning. Additionally, the context emphasizes the importance of software development skills, including familiarity with Python. It’s worth noting that there are many Python frameworks available for different purposes, such as web development (Django, Flask), data analysis (Pandas, NumPy), and more. The choice of which frameworks to learn depends on your specific goals and interests.
Predicted Grade: INCORRECT
````

````text
Example 2:
Question: Who is the founder of DeepLearning.AI and what insights did he provide in the document?
Real Answer: The founder of DeepLearning.AI is Andrew Ng and he provided insights on how to build a career in AI in the document “How to Build Your Career in AI: A Simple Guide.”
Predicted Answer: The founder of DeepLearning.AI is Andrew Ng. In the document, he provided insights on how to build a career in AI, the importance of learning foundational technical skills, working on projects, and finding a job, and the most important topics for a technical career in machine learning. He also emphasized the importance of literacy in AI-oriented coding and how it could enable individuals to harness data to make their lives richer.
Predicted Grade: CORRECT
````

````text
Example 3:
Question: Who made the statement “AI is the new electricity. It will transform and improve all areas of human life.”?
Real Answer: Andrew Ng made the statement “AI is the new electricity. It will transform and improve all areas of human life.” as seen on page 2 of the eBook “How to Build a Career in AI”.
Predicted Answer: The statement “AI is the new electricity. It will transform and improve all areas of human life.” was made by Andrew Ng.
Predicted Grade: CORRECT
````

````text
Example 4:
Question: What are the main topics covered in the eBook “How to Build a Career in AI”?
Real Answer: The eBook “How to Build a Career in AI” covers topics such as coding AI, career growth, learning technical skills, scoping successful AI projects, finding the right AI job, building a portfolio of projects, and overcoming imposter syndrome.
Predicted Answer: The eBook “How to Build a Career in AI” covers the following main topics:
````

````text
- Three Steps to Career Growth
- Learning Technical Skills for a Promising AI Career
- Should You Learn Math to Get a Job in AI?
- Scoping Successful AI Projects
- Finding Projects that Complement Your Career Goals
- Building a Portfolio of Projects that Shows Skill Progression
- A Simple Framework for Starting Your AI Job Search
- Using Informational Interviews to Find the Right Job
- Finding the Right AI Job for You
- Keys to Building a Career in AI
- Overcoming Imposter Syndrome
- Final Thoughts: Make Every Day Count.
Predicted Grade: CORRECT
````

````text
Example 5:
Question: According to the document, what is the new form of literacy that the author hopes will become as common as reading and writing?
Real Answer: The new form of literacy that the author hopes will become as common as reading and writing is coding specifically for AI.
Predicted Answer: The new form of literacy that the author hopes will become as common as reading and writing is coding specifically for AI.
Predicted Grade: CORRECT
````

````text
Example 6:
Question: What is the title of Chapter 1 in the eBook “How to Build a Career in AI”? 
Real Answer: The title of Chapter 1 is “Three Steps to Career Growth”.
Predicted Answer: The title of Chapter 1 in the eBook “How to Build a Career in AI” is “Three Steps to Career Growth.”
Predicted Grade: CORRECT
````

And looks like here it got everything correct except for the first question and it seems because our answer at the beginning of the article was very short and the right answer from the document has more details in it the evaluation was incorrect.